In [ ]:
# Problema: Evaluar cambios de esquema contra un contrato de datos para anticipar rupturas analíticas.

"""Evalúa cambios mínimos en un contrato de datos de clientes."""
from pathlib import Path
import pandas as pd

ROOT=next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir()); SOURCE=ROOT/"data/customers_valid.csv"; OUTPUT=ROOT/"submission/contract_report.csv"
REQUIRED={"customer_id","customer_name","segment","city","signup_date"}; SEGMENTS={"Basic","Premium","Corporate"}

def validate(frame):
    if not REQUIRED <= set(frame): return False, "BREAKING"
    if not set(frame.segment) <= SEGMENTS: return False, "BREAKING"
    if pd.to_datetime(frame.signup_date, format="%Y-%m-%d", errors="coerce").isna().any(): return False, "BREAKING"
    return True, "COMPATIBLE" if set(frame)-REQUIRED else "NONE"

def build_submission():
    valid=pd.DataFrame([["C1","Ana","Premium","Bogota","2026-01-03"],["C2","Luis","Basic","Cali","2026-01-04"]],columns=["customer_id","customer_name","segment","city","signup_date"])
    valid.to_csv(SOURCE,index=False)
    cases=[("valid",valid),("missing_segment",valid.drop(columns="segment")),("invalid_date",valid.assign(signup_date="not-a-date")),("unsupported_segment",valid.assign(segment="VIP")),("optional_column",valid.assign(preferred_language="es"))]
    result=[]
    for name,frame in cases:
        ok,change=validate(frame); result.append([name,"PASS" if ok else "FAIL",change,"ACCEPT" if ok else "REJECT"])
    pd.DataFrame(result,columns=["batch_name","status","change_type","action"]).to_csv(OUTPUT,index=False)
if __name__=="__main__": build_submission()